# 01 · Harmonize the Seasonal Agricultural Survey

NISR's **Seasonal Agricultural Survey (SAS)** visits about 17,000 plots every season. We use the public-use
microdata for **2020–2025, Seasons A and B** (12 seasons). 2019 is left out because its files carry no farmer ID
or plot area.

Variable *codes* change from year to year (the "was this plot irrigated?" question is `s4q13` in 2020–21,
`s4q15` in 2022–24 and `s4q14` in 2025), so `kuhira.sas` finds every field by its **label** instead.

> **Licence.** The microdata may not be redistributed. It stays in `data/raw/` (git-ignored) and this notebook
> only prints aggregates. To rerun, request each year's "Public Use Dataset" at microdata.statistics.gov.rw
> and unzip it to `data/raw/sas<year>/`.

In [1]:
import pandas as pd

from kuhira.paths import INTERIM
from kuhira.sas import harmonize_all

plots = harmonize_all()
plots.to_parquet(INTERIM / "plots.parquet")
f"{len(plots):,} crop-plot records"

'449,970 crop-plot records'

## What came out, season by season
`irrigation known` is the share of records where we can say irrigated or not. Plot-level irrigation sits in a
separate *practice* file in some years and is joined on segment, farmer and plot.

In [2]:
summary = plots.groupby(["year", "season"]).agg(
    records=("crop", "size"),
    irrigated_records=("irrigated", "sum"),
    irrigation_known=("irrigated", lambda s: s.notna().mean()),
    harvest_known=("kg", lambda s: s.notna().mean()),
    area_known=("ha", lambda s: s.notna().mean()),
    districts=("district", "nunique"),
)
summary.style.format({"irrigated_records": "{:,.0f}", "records": "{:,.0f}", "irrigation_known": "{:.1%}",
                      "harvest_known": "{:.1%}", "area_known": "{:.1%}"})

Every season covers all 30 districts and irrigation is known for at least 96% of records.

## Farmer types
SAS samples small-scale farmers on area frames and large-scale farmers from a list. The irrigation subsidy targets
small farmers, so the model uses them only.

In [3]:
plots.groupby("small").agg(records=("crop", "size"), irrigated=("irrigated", "sum")).rename(
    index={0: "large-scale", 1: "small-scale"})

,records,irrigated
small,,
large-scale,16337,4405.0
small-scale,433633,7466.0
